# Create a full pytorch dataset

In [1]:
import pandas as pd

df = pd.read_csv('raw.csv', delimiter=';')

/tmp/ipykernel_12699/386006593.py:3: DtypeWarning: Columns (163,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('raw.csv', delimiter=';')


In [2]:
df.columns
twos = 0
qcols = 0
for col in df.columns:
    if col.startswith('Q'):
        for val in df[col]:
            if val == 2:
                twos += 1

        qcols += 1
        
print(qcols)
print(twos)


36
1800


In [3]:
for col in df.columns:
    print(col)

NUM_POSTE
NOM_USUEL
LAT
LON
ALTI
AAAAMM
RR
QRR
NBRR
RR_ME
RRAB
QRRAB
RRABDAT
NBJRR1
NBJRR5
NBJRR10
NBJRR30
NBJRR50
NBJRR100
PMERM
QPMERM
NBPMERM
PMERMINAB
QPMERMINAB
PMERMINABDAT
TX
QTX
NBTX
TX_ME
TXAB
QTXAB
TXDAT
TXMIN
QTXMIN
TXMINDAT
NBJTX0
NBJTX25
NBJTX30
NBJTX35
NBJTXI20
NBJTXI27
NBJTXS32
TN
QTN
NBTN
TN_ME
TNAB
QTNAB
TNDAT
TNMAX
QTNMAX
TNMAXDAT
NBJTN5
NBJTN10
NBJTNI10
NBJTNI15
NBJTNI20
NBJTNS20
NBJTNS25
NBJGELEE
TAMPLIM
QTAMPLIM
TAMPLIAB
QTAMPLIAB
TAMPLIABDAT
NBTAMPLI
TM
QTM
NBTM
TMM
QTMM
NBTMM
NBJTMS24
TMMIN
QTMMIN
TMMINDAT
TMMAX
QTMMAX
TMMAXDAT
UNAB
QUNAB
UNABDAT
NBUN
UXAB
QUXAB
UXABDAT
NBUX
UMM
QUMM
NBUM
TSVM
QTSVM
NBTSVM
ETP
QETP
FXIAB
QFXIAB
DXIAB
QDXIAB
FXIDAT
NBJFF10
NBJFF16
NBJFF28
NBFXI
FXI3SAB
QFXI3SAB
DXI3SAB
QDXI3SAB
FXI3SDAT
NBJFXI3S10
NBJFXI3S16
NBJFXI3S28
NBFXI3S
FXYAB
QFXYAB
DXYAB
QDXYAB
FXYABDAT
NBJFXY8
NBJFXY10
NBJFXY15
NBFXY
FFM
QFFM
NBFFM
INST
QINST
NBINST
NBSIGMA0
NBSIGMA20
NBSIGMA80
GLOT
QGLOT
NBGLOT
DIFT
QDIFT
NBDIFT
DIRT
QDIRT
NBDIRT
HNEIGEFTOT
QHNEIGEFTOT
H

In [4]:
# Cell 1 — Build monthly date and filter to 2001-01 through 2025-12

import pandas as pd
import numpy as np

df_proc = df.copy()

# Robust monthly date from AAAAMM (expected format yyyymm, sometimes numeric)
df_proc["AAAAMM"] = df_proc["AAAAMM"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)

df_proc["month"] = pd.to_datetime(
    df_proc["AAAAMM"],
    format="%Y%m",
    errors="coerce"
).dt.to_period("M")

# Keep only the modelling period: 2001-01 to 2025-12 inclusive
start_month = pd.Period("2001-01", freq="M")
end_month = pd.Period("2025-12", freq="M")

rows_before = len(df_proc)
df_proc = df_proc[
    (df_proc["month"] >= start_month) &
    (df_proc["month"] <= end_month)
].copy()

# Basic sorting for later station-wise processing
df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

print("Rows before filter:", rows_before)
print("Rows after filter:", len(df_proc))
print("Unique NUM_POSTE:", df_proc["NUM_POSTE"].nunique())
print("Min month:", df_proc["month"].min())
print("Max month:", df_proc["month"].max())

df_proc[["NUM_POSTE", "AAAAMM", "month"]].head()

Rows before filter: 3262791
Rows after filter: 1044600
Unique NUM_POSTE: 5329
Min month: 2001-01
Max month: 2025-12


,NUM_POSTE,AAAAMM,month
0,1014002.0,200403,2004-03
1,1014002.0,200404,2004-04
2,1014002.0,200405,2004-05
3,1014002.0,200406,2004-06
4,1014002.0,200407,2004-07


In [5]:
1+1

2

In [7]:
# Cell 1.1 — Missingness report by column

import pandas as pd
import numpy as np

df_check = df_proc.copy()

missing_report = (
    df_check.isna()
    .sum()
    .to_frame("n_missing")
)

missing_report["pct_missing"] = missing_report["n_missing"] / len(df_check) * 100
missing_report["dtype"] = df_check.dtypes.astype(str)

missing_report = missing_report.sort_values(
    by=["pct_missing", "n_missing"],
    ascending=False
)

print("Total rows:", len(df_check))
print("Total columns:", df_check.shape[1])

# Show the worst offenders
display(missing_report.head(50))

# Optional: only columns with any missingness
cols_with_missing = missing_report[missing_report["n_missing"] > 0]
print("\nColumns with missing values:", len(cols_with_missing))

# Optional: quick view of the main model columns only
model_cols = [c for c in df_check.columns if c not in ["month"]]
model_missing = (
    df_check[model_cols]
    .isna()
    .sum()
    .to_frame("n_missing")
)
model_missing["pct_missing"] = model_missing["n_missing"] / len(df_check) * 100
model_missing = model_missing.sort_values("pct_missing", ascending=False)

print("\nTop missing model columns:")
display(model_missing.head(50))

Total rows: 1044600
Total columns: 167


,n_missing,pct_missing,dtype
code_departement,1044600,100.000000,float64
nom_departement,1044600,100.000000,object
code_region,1044600,100.000000,float64
nom_region,1044600,100.000000,object
DIFT,1043756,99.919204,float64
QDIFT,1043756,99.919204,float64
TX_ME,1043316,99.877082,float64
TN_ME,1043313,99.876795,float64
DIRT,1042838,99.831323,float64
QDIRT,1042838,99.831323,float64



Columns with missing values: 160

Top missing model columns:


,n_missing,pct_missing
code_region,1044600,100.000000
nom_region,1044600,100.000000
code_departement,1044600,100.000000
nom_departement,1044600,100.000000
QDIFT,1043756,99.919204
DIFT,1043756,99.919204
TX_ME,1043316,99.877082
TN_ME,1043313,99.876795
QDIRT,1042838,99.831323
DIRT,1042838,99.831323


In [8]:
# Cell 2 — Clean columns, process Q flags, keep >=97% complete features
import pandas as pd
import numpy as np

df_proc = df_proc.copy()

# -----------------------------
# 0) Drop textual / metadata columns we do not want
# -----------------------------
drop_text_cols = [
    "NOM_USUEL", "nom_usuel",
    "nom_region", "NOM_REGION",
    "nom_departement", "nom_department", "NOM_DEPARTEMENT",
    "code_region", "CODE_REGION",
]
df_proc = df_proc.drop(columns=[c for c in drop_text_cols if c in df_proc.columns], errors="ignore")

# -----------------------------
# 1) Clean NUM_POSTE and derive code_departement
# -----------------------------
df_proc["NUM_POSTE"] = (
    df_proc["NUM_POSTE"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
)

def derive_code_departement(num_poste):
    s = "".join(ch for ch in str(num_poste) if ch.isdigit())
    if len(s) == 0:
        return pd.NA
    s = s.zfill(8)
    # Overseas / special codes
    if s.startswith(("97", "98", "99")):
        return s[:3]
    return s[:2]

df_proc["code_departement"] = df_proc["NUM_POSTE"].apply(derive_code_departement).astype("string")

# -----------------------------
# 2) Target column
# -----------------------------
target_col = "TMM"
if target_col not in df_proc.columns:
    raise ValueError(f"Target column {target_col} not found in dataframe.")

# Ensure month exists from Cell 1
if "month" not in df_proc.columns:
    raise ValueError("month column missing. Run Cell 1 first.")

# Sort and remove duplicate station-month rows if any
df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

n_dupes = df_proc.duplicated(subset=["NUM_POSTE", "month"]).sum()
print("Duplicate NUM_POSTE-month rows before:", int(n_dupes))

if n_dupes > 0:
    df_proc = (
        df_proc
        .sort_values(["NUM_POSTE", "month"])
        .drop_duplicates(subset=["NUM_POSTE", "month"], keep="first")
        .copy()
    )

df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

# -----------------------------
# 3) Process all Q columns:
#    if Q == 2 -> impute paired value from same month last/next year
#    then Q becomes binary: 1 if flagged, else 0
# -----------------------------
q_cols = [c for c in df_proc.columns if c.startswith("Q") and len(c) > 1]

def seasonal_impute_for_station_group(g, value_col, q_col):
    g = g.sort_values("month").copy()

    # numeric conversion for the value column
    g[value_col] = pd.to_numeric(g[value_col], errors="coerce")

    months = pd.PeriodIndex(g["month"], freq="M")
    values = g[value_col].to_numpy(dtype="float64")

    lookup = pd.Series(values, index=months)

    prev_vals = lookup.reindex(months - 12).to_numpy()
    next_vals = lookup.reindex(months + 12).to_numpy()

    both_available = ~pd.isna(prev_vals) & ~pd.isna(next_vals)
    only_prev = ~pd.isna(prev_vals) & pd.isna(next_vals)
    only_next = pd.isna(prev_vals) & ~pd.isna(next_vals)

    fill_values = np.full(len(g), np.nan, dtype="float64")
    fill_values[both_available] = (prev_vals[both_available] + next_vals[both_available]) / 2.0
    fill_values[only_prev] = prev_vals[only_prev]
    fill_values[only_next] = next_vals[only_next]

    q_mask = pd.to_numeric(g[q_col], errors="coerce").eq(2).fillna(False).to_numpy()

    # Fill only where the flag says Q == 2 and a fill value exists
    can_fill = q_mask & ~pd.isna(fill_values)
    g.loc[can_fill, value_col] = fill_values[can_fill]

    # Convert Q column to binary:
    # 1 = was flagged (Q == 2), 0 = everything else
    g[q_col] = 0
    g.loc[q_mask, q_col] = 1

    return g

processed = 0
skipped = []

for q_col in q_cols:
    value_col = q_col[1:]  # QRR -> RR, QTMM -> TMM, etc.

    if value_col not in df_proc.columns:
        skipped.append((q_col, value_col))
        continue

    # Apply station-wise to preserve seasonality
    df_proc = (
        df_proc
        .groupby("NUM_POSTE", group_keys=False)
        .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
        .reset_index(drop=True)
    )

    processed += 1

print("\nQ columns found:", len(q_cols))
print("Q/value pairs processed:", processed)
print("Q columns skipped because base column missing:", len(skipped))
if skipped:
    print("Examples skipped:", skipped[:10])

# -----------------------------
# 4) Keep only columns that are >=97% complete
#    - always keep identifiers / target / month
#    - keep Q columns only if their base variable is kept
# -----------------------------
missing_pct = df_proc.isna().mean() * 100

mandatory_cols = ["NUM_POSTE", "code_departement", "month", target_col]
exclude_from_threshold = set(mandatory_cols + ["AAAAMM"])

# Base columns = non-Q columns eligible by completeness
base_candidate_cols = [
    c for c in df_proc.columns
    if c not in exclude_from_threshold
    and not c.startswith("Q")
]

keep_base_cols = []
for col in base_candidate_cols:
    if pd.api.types.is_numeric_dtype(df_proc[col]) or col in ["NUM_POSTE", "code_departement"]:
        if missing_pct.get(col, 100.0) <= 3.0:
            keep_base_cols.append(col)

# Keep Q columns only if their base variable is kept
keep_q_cols = []
for q_col in q_cols:
    base_col = q_col[1:]
    if base_col in keep_base_cols and q_col in df_proc.columns:
        keep_q_cols.append(q_col)

# Make sure target is kept
if target_col not in keep_base_cols:
    keep_base_cols.append(target_col)

print("\nBase columns kept:", len(keep_base_cols))
print("Q columns kept:", len(keep_q_cols))

# -----------------------------
# 5) Impute remaining missing values for kept base numeric columns
#    using same month last/next year.
#    No extra imputation flags are created.
# -----------------------------
def seasonal_impute_missing(df_in, col):
    df_out = df_in.copy()
    df_out[col] = pd.to_numeric(df_out[col], errors="coerce")

    for station, idx in df_out.groupby("NUM_POSTE").groups.items():
        idx = list(idx)
        g = df_out.loc[idx].sort_values("month").copy()

        months = pd.PeriodIndex(g["month"], freq="M")
        values = g[col].to_numpy(dtype="float64")
        lookup = pd.Series(values, index=months)

        prev_vals = lookup.reindex(months - 12).to_numpy()
        next_vals = lookup.reindex(months + 12).to_numpy()

        both_available = ~pd.isna(prev_vals) & ~pd.isna(next_vals)
        only_prev = ~pd.isna(prev_vals) & pd.isna(next_vals)
        only_next = pd.isna(prev_vals) & ~pd.isna(next_vals)

        fill_values = np.full(len(g), np.nan, dtype="float64")
        fill_values[both_available] = (prev_vals[both_available] + next_vals[both_available]) / 2.0
        fill_values[only_prev] = prev_vals[only_prev]
        fill_values[only_next] = next_vals[only_next]

        missing_mask = g[col].isna().to_numpy()
        can_fill = missing_mask & ~pd.isna(fill_values)

        df_out.loc[g.index[can_fill], col] = fill_values[can_fill]

    return df_out

# Only impute retained base numeric columns, not identifiers or Q flags
impute_cols = [
    c for c in keep_base_cols
    if c not in ["NUM_POSTE", "code_departement", "month", target_col]
    and c in df_proc.columns
    and pd.api.types.is_numeric_dtype(df_proc[c])
]

# IMPORTANT: do NOT impute the target
# We'll keep TMM as the target and drop rows where it is missing.

print("\nImputing retained base numeric columns:")
for col in impute_cols:
    before = int(df_proc[col].isna().sum())
    if before > 0:
        df_proc = seasonal_impute_missing(df_proc, col)
    after = int(df_proc[col].isna().sum())
    print(f"{col}: missing before={before}, after={after}")

# -----------------------------
# 6) Final retained columns
# -----------------------------
imputed_cols = [c for c in df_proc.columns if c.endswith("_imputed")]  # should be empty here
final_cols = list(dict.fromkeys(
    mandatory_cols
    + keep_base_cols
    + keep_q_cols
    + imputed_cols
))

# Remove AAAAMM if present
final_cols = [c for c in final_cols if c in df_proc.columns and c != "AAAAMM"]

df_proc = df_proc[final_cols].copy()

# -----------------------------
# 7) Drop any rows with remaining missing in kept columns
#    This includes dropping rows where the target TMM is missing.
# -----------------------------
remaining_missing_rows = df_proc.isna().any(axis=1).sum()
print("\nRows with remaining missing in final kept columns:", int(remaining_missing_rows))

if remaining_missing_rows > 0:
    df_proc = df_proc.loc[~df_proc.isna().any(axis=1)].copy()

df_proc = df_proc.sort_values(["NUM_POSTE", "month"]).reset_index(drop=True)

print("\nFinal shape:", df_proc.shape)
print("Final number of columns:", df_proc.shape[1])
print("Min month:", df_proc["month"].min())
print("Max month:", df_proc["month"].max())
print("Unique NUM_POSTE:", df_proc["NUM_POSTE"].nunique())

print("\nFinal missing counts in kept columns (should be all zero):")
display(df_proc.isna().sum().sort_values(ascending=False).head(30))

Duplicate NUM_POSTE-month rows before: 0


/tmp/ipykernel_12699/2048032608.py:124: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
/tmp/ipykernel_12699/2048032608.py:124: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: seasonal_impute_for_station_group(g, value_col, q_col))
/tmp/ipykernel_12699/2048032608.py:124: FutureWarning: DataFrameGroupBy.apply operated on the grouping


Q columns found: 36
Q/value pairs processed: 36
Q columns skipped because base column missing: 0

Base columns kept: 31
Q columns kept: 2

Imputing retained base numeric columns:
LAT: missing before=0, after=0
LON: missing before=0, after=0
ALTI: missing before=0, after=0
RR: missing before=20923, after=15731
NBRR: missing before=2908, after=312
RRAB: missing before=20895, after=15729
RRABDAT: missing before=23516, after=16412
NBJRR1: missing before=20901, after=15729
NBJRR5: missing before=20901, after=15729
NBJRR10: missing before=20901, after=15729
NBJRR30: missing before=23312, after=17099
NBJRR50: missing before=23312, after=17099
NBJRR100: missing before=23312, after=17099
NBPMERM: missing before=2846, after=284
NBTX: missing before=3167, after=543
NBTN: missing before=3174, after=543
NBTAMPLI: missing before=3184, after=542
NBTM: missing before=3223, after=548
NBTMM: missing before=2854, after=284
NBUN: missing before=3067, after=329
NBUX: missing before=2931, after=328
NBUM: m

NUM_POSTE           0
code_departement    0
month               0
TMM                 0
LAT                 0
LON                 0
ALTI                0
RR                  0
NBRR                0
RRAB                0
RRABDAT             0
NBJRR1              0
NBJRR5              0
NBJRR10             0
NBJRR30             0
NBJRR50             0
NBJRR100            0
NBPMERM             0
NBTX                0
NBTN                0
NBTAMPLI            0
NBTM                0
NBTMM               0
NBUN                0
NBUX                0
NBUM                0
NBFXI               0
NBFXY               0
NBFFM               0
NBINST              0
dtype: int64